In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
import statsmodels.graphics.tsaplots
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import statistics

In [2]:
# Get data between specified dates
def filter_times(df, time1, time2):
    return df.apply(lambda x: (time1 <= x['begin_time']) and (x['begin_time'] <= time2) , axis=1)

In [3]:
data = pd.read_csv('NP15_rt_series.csv')
data['begin_time'] = pd.to_datetime(data['begin_time'])
data_15min = data[data['begin_time'].apply(lambda x : x.minute%15 == 0)].copy()
data_15min

,begin_time,NP-15 LMP,NP-15 LMP 24hrdiffs,NP-15 LMP Prev_day_price
0,2021-01-01 00:00:00,30.76070,NaN,NaN
3,2021-01-01 00:15:00,31.35044,NaN,NaN
6,2021-01-01 00:30:00,32.45147,NaN,NaN
9,2021-01-01 00:45:00,29.98897,NaN,NaN
12,2021-01-01 01:00:00,29.89887,NaN,NaN
...,...,...,...,...
324012,2024-01-31 01:00:00,41.87905,6.91134,34.96771
324015,2024-01-31 01:15:00,39.04780,5.16035,33.88745
324018,2024-01-31 01:30:00,39.96866,7.25526,32.71340
324021,2024-01-31 01:45:00,38.67257,6.70449,31.96808


In [4]:
exog = pd.read_csv('NP15_exog.csv')
exog['begin_time'] = pd.to_datetime(exog['begin_time'])
exog

,begin_time,load_actual,load_forecast,solar_forecast,wind_forecast,solar_actual,wind_actual,nat_gas_price
0,2021-03-12 00:00:00,9914.0,10102.7200,0.0,327.9700,-2.69832,526.78168,2.65
1,2021-03-12 00:15:00,9914.0,10044.1825,0.0,310.9625,-2.69832,526.78168,2.65
2,2021-03-12 00:30:00,9914.0,9985.6450,0.0,293.9550,-2.69832,526.78168,2.65
3,2021-03-12 00:45:00,9914.0,9927.1075,0.0,276.9475,-2.69832,526.78168,2.65
4,2021-03-12 01:00:00,9827.0,9868.5700,0.0,259.9400,-2.66009,405.97186,2.65
...,...,...,...,...,...,...,...,...
133528,2024-12-31 22:00:00,10226.0,10831.4600,0.0,77.4600,-6.26466,54.79276,3.40
133529,2024-12-31 22:15:00,10226.0,10682.2425,0.0,80.2975,-6.26466,54.79276,3.40
133530,2024-12-31 22:30:00,10226.0,10533.0250,0.0,83.1350,-6.26466,54.79276,3.40
133531,2024-12-31 22:45:00,10226.0,10383.8075,0.0,85.9725,-6.26466,54.79276,3.40


In [5]:
data_comb = pd.merge(data_15min, exog, how='inner', on='begin_time')
data_comb

,begin_time,NP-15 LMP,NP-15 LMP 24hrdiffs,NP-15 LMP Prev_day_price,load_actual,load_forecast,solar_forecast,wind_forecast,solar_actual,wind_actual,nat_gas_price
0,2021-03-12 00:00:00,34.56084,-21.12893,55.68977,9914.0,10102.7200,0.0,327.9700,-2.69832,526.78168,2.65
1,2021-03-12 00:15:00,35.35549,-25.15432,60.50981,9914.0,10044.1825,0.0,310.9625,-2.69832,526.78168,2.65
2,2021-03-12 00:30:00,34.51972,-22.99046,57.51018,9914.0,9985.6450,0.0,293.9550,-2.69832,526.78168,2.65
3,2021-03-12 00:45:00,32.77203,-14.96161,47.73364,9914.0,9927.1075,0.0,276.9475,-2.69832,526.78168,2.65
4,2021-03-12 01:00:00,33.10504,-18.19313,51.29817,9827.0,9868.5700,0.0,259.9400,-2.66009,405.97186,2.65
...,...,...,...,...,...,...,...,...,...,...,...
101284,2024-01-31 01:00:00,41.87905,6.91134,34.96771,9273.0,9058.9400,0.0,110.5300,-4.08653,133.86907,2.19
101285,2024-01-31 01:15:00,39.04780,5.16035,33.88745,9273.0,9009.9400,0.0,112.8500,-4.08653,133.86907,2.19
101286,2024-01-31 01:30:00,39.96866,7.25526,32.71340,9273.0,8960.9400,0.0,115.1700,-4.08653,133.86907,2.19
101287,2024-01-31 01:45:00,38.67257,6.70449,31.96808,9273.0,8911.9400,0.0,117.4900,-4.08653,133.86907,2.19


In [6]:
data_comb['load_diffs']=data_comb['load_actual']-data_comb['load_forecast']
data_comb['load_diffs_prevtime']=data_comb['load_diffs'].shift(1)
data_comb['solar_diffs']=data_comb['solar_actual']-data_comb['solar_forecast']
data_comb['solar_diffs_prevtime']=data_comb['solar_diffs'].shift(1)
data_comb['wind_diffs']=data_comb['wind_actual']-data_comb['wind_forecast']
data_comb['wind_diffs_prevtime']=data_comb['wind_diffs'].shift(1)
data_comb['load_prevtime'] = data_comb['load_actual'].shift(1)
data_comb['solar_prevtime'] = data_comb['solar_actual'].shift(1)
data_comb['wind_prevtime'] = data_comb['wind_actual'].shift(1)

In [7]:
data_train=data_comb.iloc[:106826]
data_train

,begin_time,NP-15 LMP,NP-15 LMP 24hrdiffs,NP-15 LMP Prev_day_price,load_actual,load_forecast,solar_forecast,wind_forecast,solar_actual,wind_actual,nat_gas_price,load_diffs,load_diffs_prevtime,solar_diffs,solar_diffs_prevtime,wind_diffs,wind_diffs_prevtime,load_prevtime,solar_prevtime,wind_prevtime
0,2021-03-12 00:00:00,34.56084,-21.12893,55.68977,9914.0,10102.7200,0.0,327.9700,-2.69832,526.78168,2.65,-188.7200,NaN,-2.69832,NaN,198.81168,NaN,NaN,NaN,NaN
1,2021-03-12 00:15:00,35.35549,-25.15432,60.50981,9914.0,10044.1825,0.0,310.9625,-2.69832,526.78168,2.65,-130.1825,-188.7200,-2.69832,-2.69832,215.81918,198.81168,9914.0,-2.69832,526.78168
2,2021-03-12 00:30:00,34.51972,-22.99046,57.51018,9914.0,9985.6450,0.0,293.9550,-2.69832,526.78168,2.65,-71.6450,-130.1825,-2.69832,-2.69832,232.82668,215.81918,9914.0,-2.69832,526.78168
3,2021-03-12 00:45:00,32.77203,-14.96161,47.73364,9914.0,9927.1075,0.0,276.9475,-2.69832,526.78168,2.65,-13.1075,-71.6450,-2.69832,-2.69832,249.83418,232.82668,9914.0,-2.69832,526.78168
4,2021-03-12 01:00:00,33.10504,-18.19313,51.29817,9827.0,9868.5700,0.0,259.9400,-2.66009,405.97186,2.65,-41.5700,-13.1075,-2.66009,-2.69832,146.03186,249.83418,9914.0,-2.69832,526.78168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101284,2024-01-31 01:00:00,41.87905,6.91134,34.96771,9273.0,9058.9400,0.0,110.5300,-4.08653,133.86907,2.19,214.0600,402.3100,-4.08653,-4.07035,23.33907,-1.38558,9530.0,-4.07035,109.43942
101285,2024-01-31 01:15:00,39.04780,5.16035,33.88745,9273.0,9009.9400,0.0,112.8500,-4.08653,133.86907,2.19,263.0600,214.0600,-4.08653,-4.08653,21.01907,23.33907,9273.0,-4.08653,133.86907
101286,2024-01-31 01:30:00,39.96866,7.25526,32.71340,9273.0,8960.9400,0.0,115.1700,-4.08653,133.86907,2.19,312.0600,263.0600,-4.08653,-4.08653,18.69907,21.01907,9273.0,-4.08653,133.86907
101287,2024-01-31 01:45:00,38.67257,6.70449,31.96808,9273.0,8911.9400,0.0,117.4900,-4.08653,133.86907,2.19,361.0600,312.0600,-4.08653,-4.08653,16.37907,18.69907,9273.0,-4.08653,133.86907


In [8]:
def crossValidateFinal(year, n_splits, test_size, seas_order, non_seas_order):
    ts_split = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    mse_list = []
    mse_base_list = []

    print(f"Seasonal order - {seas_order}, Non-seasonal order - {non_seas_order}")

    for tod in range(0, 2):
      for j in range(3,13, 3):
          time1 = datetime.datetime(year, j, 1, 0, 0, 0)
          time2 = datetime.datetime(year, j, 28, (12 + (6*tod)), 0, 0)
          df = data_train[filter_times(data_train, time1, time2)].copy().reset_index(drop=True)
          for i, (train_index, test_index) in enumerate(ts_split.split(df)):
              model = SARIMAX(endog=df['NP-15 LMP'].loc[train_index], exog=df[['nat_gas_price', 'load_diffs_prevtime', 'solar_diffs_prevtime', 'wind_diffs_prevtime', 'solar_prevtime', 'wind_prevtime', 'load_prevtime']].loc[train_index], trend='c', order=non_seas_order, seasonal_order=seas_order)
              model_fit = model.fit()
              forecast = model_fit.forecast(test_size, exog=df[['nat_gas_price', 'load_diffs_prevtime', 'solar_diffs_prevtime', 'wind_diffs_prevtime', 'solar_prevtime', 'wind_prevtime', 'load_prevtime']].loc[test_index]).to_frame(name='predicted')
              eval_df = pd.merge(df[['NP-15 LMP', 'NP-15 LMP Prev_day_price']].loc[test_index].copy(), forecast, left_index=True, right_index=True, how='inner')
              #eval_df = eval_df[eval_df.apply(lambda x: (x['NP-15 LMP'].isna()==False) and (x['NP-15 LMP Prev_day_price'].isna()==False))]
              eval_df = eval_df[(eval_df['NP-15 LMP'].isna()==False)]
              eval_df = eval_df[(eval_df['NP-15 LMP Prev_day_price'].isna()==False)]
              if eval_df.empty == False:
                  mse = mean_squared_error(eval_df['NP-15 LMP'], eval_df['predicted'])
                  mse_base = mean_squared_error(eval_df['NP-15 LMP'], eval_df['NP-15 LMP Prev_day_price'])
                  mse_list.append(mse)
                  mse_base_list.append(mse_base)
                  #print(f"mse for validation from time {df['begin_time'].loc[test_index[0]]} to {df['begin_time'].loc[test_index[-1]]} , fold {i} = {mse}. Baseline = {mse_base}")
              #print(model_fit.summary())
              #mse = mean_squared_error(df['NP-15 LMP'].loc[test_index], model_fit.forecast(test_size))
              #mse_list.append(mse)
              #data_predicted = df.join(pd.concat([df['NP-15 LMP'].loc[train_index].tail(test_size), model_fit.forecast(test_size, exog=df[['Natural_gas_price', 'load_diffs_prevtime', 'solar_diffs_prevtime', 'wind_diffs_prevtime', 'solar_prevtime', 'wind_prevtime', 'load_prevtime']].loc[test_index])]).to_frame(name='predicted'), how='inner')
              #plt.figure(figsize=(18, 4))
              #plt.plot(data_predicted['begin_time'], data_predicted['NP-15 LMP Prev_day_price'] , label='Previous day price')
              #plt.plot(data_predicted['begin_time'], data_predicted['NP-15 LMP'] , label='Actual price')
              #plt.plot(data_predicted['begin_time'], data_predicted['predicted'] , label='Predicted')
              #plt.xticks(rotation=90)
              #plt.legend()
              #plt.show()
      print(f"tod {tod} done.")

    print(f"Validation error for {year} = {statistics.fmean(mse_list)}, baseline = {statistics.fmean(mse_base_list)}")

In [84]:
crossValidateFinal(2023, 3, 8, (0,1,0, 96), (0,0,0))
crossValidateFinal(2023, 3, 8, (0,1,0, 96), (0,1,0))
#crossValidateFinal(2023, 3, 8, (0,1,0, 96), (1,0,0)) Fails to converge

Seasonal order - (0, 1, 0, 96), Non-seasonal order - (0, 0, 0)
tod 0 done.
tod 1 done.
Validation error for 2023 = 287.23965325761566, baseline = 1950.9388681499104
Seasonal order - (0, 1, 0, 96), Non-seasonal order - (0, 1, 0)
tod 0 done.
tod 1 done.
Validation error for 2023 = 466.9753617546159, baseline = 1950.9388681499104
Seasonal order - (0, 1, 0, 96), Non-seasonal order - (1, 0, 0)


/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


tod 0 done.


KeyboardInterrupt: 

In [90]:
crossValidateFinal(2023, 3, 8, (0,1,0, 96), (1,1,0))

Seasonal order - (0, 1, 0, 96), Non-seasonal order - (1, 1, 0)
tod 0 done.
tod 1 done.
Validation error for 2023 = 434.8647855877363, baseline = 2007.4000075839724


With refined exog variables

In [103]:
crossValidateFinal(2023, 3, 8, (0,1,0, 96), (0,0,0))

Seasonal order - (0, 1, 0, 96), Non-seasonal order - (0, 0, 0)


/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


tod 0 done.
tod 1 done.
Validation error for 2023 = 286.06939913973105, baseline = 2007.4000075839724


In [10]:
crossValidateFinal(2023, 3, 8, (0,1,0, 96), (1,0,0))

Seasonal order - (0, 1, 0, 96), Non-seasonal order - (1, 0, 0)
tod 0 done.


/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


tod 1 done.
Validation error for 2023 = 207.10313863531505, baseline = 2007.4000075839724


In [9]:
crossValidateFinal(2023, 3, 8, (0,1,0, 96), (0,1,0))

Seasonal order - (0, 1, 0, 96), Non-seasonal order - (0, 1, 0)
tod 0 done.
tod 1 done.
Validation error for 2023 = 208.62434433153922, baseline = 2007.4000075839724
